<a href="https://colab.research.google.com/github/guilhermelaviola/SRTTranslator/blob/main/SubtitleTranslatorWithMarianMT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformers torch pysrt sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pysrt: filename=pysrt-1.1.2-py3-none-any.whl size=13443 sha256=2446a5d157fe08281af7e007caa915e185ac64f5f36679852e5f9fea3f89434c
  Stored in directory: /root/.cache/pip/wheels/6a/36/54/2aa8dc961885dfa7b0ebd45a57505f25039d79b4ea0fd9f29d
Successfully built pysrt


In [4]:
# Importing all the necessary libraries and resources:
import pysrt
import torch
import re
import textwrap
from transformers import MarianMTModel, MarianTokenizer

## **Full working script: English to Brazilian Portuguese**

In [5]:
# Loading MarianMT model and tokenizer:
model_name = 'Helsinki-NLP/opus-mt-en-pt'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(device)

# Reading the SRT file:
subs = pysrt.open('input.srt', encoding='utf-8')

# Cleaning subtitle text:
def clean_text(text):
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\[.*?\]', '', text)
    return text.replace('\n', ' ').strip()

# Subtitle formatting according to Brazilian subtitle standards: Max 42 characters per line; Max 2 lines:
def format_subtitle(text, width=42):
    lines = textwrap.wrap(text, width)
    return '\n'.join(lines[:2])

texts = [clean_text(sub.text) for sub in subs]

inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

translated = model.generate(**inputs)

translations = [
    format_subtitle(tokenizer.decode(t, skip_special_tokens=True))
    for t in translated
]

for sub, text in zip(subs, translations):
    sub.text = text

subs.save('output_pt-BR.srt', encoding='utf-8')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


OSError: Helsinki-NLP/opus-mt-en-pt is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [ ]:
from huggingface_hub import notebook_login

notebook_login()